# Reproducing "Reconstructing, Cleaning, and Measuring the Coverage of a Decade of Colorado's Public Risk-Limiting Audit Data"

This notebook calls the `corla_results` library against the bundled
`cross_election.db` to reproduce every table in the paper (see
`paper.md`), in paper section order. See `DATABASE.md` for the
database's schema and provenance, and each `corla_results` module's own
docstring for exactly what is and isn't re-derived here (a few small,
explicitly-documented reference values -- Table 2 and Table 3's full
figures, and 2018 general's row in Table 4 -- are reported as verified
constants rather than recomputed, because reproducing them from this
database alone would require machinery outside this package's scope;
see each module's docstring for the specifics).

In [1]:
import sqlite3
from corla_results import coverage, discrepancies, verification

conn = sqlite3.connect("cross_election.db")

## Table 1 -- Contest coverage classification by election, and by county span (§4.2)

In [2]:
rows = coverage.coverage_by_election(conn)
print(f"{'Election':20s} {'Total':>6s} {'Targeted':>9s} {'Opport.':>8s} {'NoRecord':>9s} {'Single':>7s} {'Partial':>8s} {'Statewide':>9s}")
for r in rows:
    print(f"{r.election_key:20s} {r.total_contests:6d} {r.directly_targeted:9d} {r.opportunistically_named:8d} {r.no_comparison_record:9d} {r.single_county:7d} {r.partial_multi_county:8d} {r.statewide:9d}")
t = coverage.coverage_totals(conn)
print(f"{t.election_key:20s} {t.total_contests:6d} {t.directly_targeted:9d} {t.opportunistically_named:8d} {t.no_comparison_record:9d} {t.single_county:7d} {t.partial_multi_county:8d} {t.statewide:9d}")

Election              Total  Targeted  Opport.  NoRecord  Single  Partial Statewide
2019-statewide          576        52      498        26     457      117         2
2020-general            652        60      520        72     510      142         0
2020-presidential         2         2        0         0       0        0         2
2020-stateprimary       372        34      304        34     282       87         3
2021-coordinated        666        54      588        24     537      127         2
2022-general            961        43      871        47     801      135        25
2022-primary            651        38      604         9     543       96        12
2023-coordinated        661        43      542        76     511      148         2
2024-general            721        63      597        61     555      142        24
2025-coordinated        640        62      541        37     515      124         1
2026-primary            620        26      580        14     512       97   

## Table 2 -- Selection-reproduction results by election (§5.2)

`verification.sha256_random_numbers` is CORLA's own real selection
algorithm, confirmed by direct reading of its published source code.
Reproducing the full sweep below requires reconstructing each contest's
exact ballot "domain" (population size and ordering) the algorithm
indexes into, which for multi-county/statewide contests is not yet
re-derived from this database alone -- see `verification.py`'s
docstring. The table below reports the paper's own already-verified
sweep results.

In [3]:
print(f"{'Election':20s} {'County-level':>14s} {'Statewide':>16s}")
for r in verification.TABLE_2:
    print(f"{r.election_key:20s} {r.county_level_match:>14s} {r.statewide_match:>16s}")

Election               County-level        Statewide
2019-statewide                63/63              1/1
2020-general                  61/63    0/0 (blocked)
2020-presidential     0/0 (blocked)    0/0 (blocked)
2020-stateprimary             88/88              1/1
2021-coordinated              62/64    0/0 (blocked)
2022-primary                  78/80              2/2
2022-general                  62/63    0/6 (blocked)
2023-coordinated              62/63              1/1
2024-general                  62/63              2/2
2025-coordinated              72/73              2/2
2026-primary                112/112              1/1


### CORLA's own PRNG algorithm, demonstrated directly

`sha256_random_numbers(seed, count, domain_size)`: for each 1-based
index *i*, hash `"{seed},{i}"` with SHA-256, interpret the digest as a
big-endian integer, and take `(int mod domain_size) + 1`. This is the
exact algorithm run here, not an approximation.

In [4]:
example_picks = verification.sha256_random_numbers(seed="12345", count=5, domain_size=1000)
example_picks

[953, 84, 716, 261, 754]

## Table 3 -- Typed-discrepancy validation, 477 formally targeted contests (§5.3)

In [5]:
print("Independently derived:    ", verification.TABLE_3_INDEPENDENTLY_DERIVED)
print("CORLA's own reported count:", verification.TABLE_3_CORLA_REPORTED)

Independently derived:     TypedDiscrepancyValidationRow(o2=18, o1=8, u1=12, u2=8, total=46)
CORLA's own reported count: TypedDiscrepancyValidationRow(o2=18, o1=8, u1=12, u2=8, total=46)


## Table 4 -- Discrepancy counts and rates by election (§6.2)

In [6]:
rows = discrepancies.discrepancy_types_by_election(conn)
print(f"{'Election':20s} {'o2':>4s} {'o1':>4s} {'u1':>4s} {'u2':>4s} {'Total':>6s} {'Examined':>10s} {'Rate':>9s}")
for r in rows:
    print(f"{r.election_key:20s} {r.o2:4d} {r.o1:4d} {r.u1:4d} {r.u2:4d} {r.total:6d} {r.examined:10d} {r.rate*100:8.4f}%")
t = discrepancies.discrepancy_types_overall(conn)
print(f"{t.election_key:20s} {t.o2:4d} {t.o1:4d} {t.u1:4d} {t.u2:4d} {t.total:6d} {t.examined:10d} {t.rate*100:8.4f}%")

Election               o2   o1   u1   u2  Total   Examined      Rate
2018-general            6    3    0    3     12     164796   0.0073%
2020-general           59   68   71   62    260     234333   0.1110%
2020-presidential       0    0    0    0      0        155   0.0000%
2020-stateprimary      16    4    4   16     40      61546   0.0650%
2021-coordinated        5   11   12    5     33      60597   0.0545%
2022-primary            6    8    8    8     30     118827   0.0252%
2023-coordinated        5    3    9    2     19      35486   0.0535%
2024-general           47   52   65   61    225     149272   0.1507%
2025-coordinated        4    6    4    4     18      47706   0.0377%
2026-primary            5    0    0    2      7      92536   0.0076%
Overall               153  155  173  163    644     965254   0.0667%


## Table 5 -- Reconciliation of the independently re-derived discrepancy-report row counts (§6.4)

In [7]:
print(f"{'Election':20s} {'Re-derived':>11s} {'Published':>10s} {'Gap':>5s}")
total_r = total_p = 0
for r in discrepancies.reconciliation_by_election(conn):
    print(f"{r.election_key:20s} {r.rederived:11d} {r.previously_published:10d} {r.gap:5d}")
    total_r += r.rederived
    total_p += r.previously_published
print(f"{'Total':20s} {total_r:11d} {total_p:10d} {total_r - total_p:5d}")

Election              Re-derived  Published   Gap
2020-stateprimary            119        122    -3
2020-general                 276        280    -4
2021-coordinated              38         44    -6
2022-primary                  86         93    -7
2023-coordinated              22         22     0
2024-general                 235        235     0
2025-coordinated              22         22     0
2026-primary                  16         16     0
Total                        814        834   -20


## Summary

Every number printed above should match the corresponding table in
`paper.md` exactly. If it doesn't, please open an issue -- that would
mean either this database or this notebook has drifted from what the
paper actually reports.